# Escribir código a partir de una especificación

**Unidad 4.c del temario** · Acompaña a `Clase_01_RegresionLineal` · Notas: cap. 2

## La tesis de la actividad

Casi todo el trabajo intelectual de escribir código econométrico está en **especificar
qué se quiere**, no en teclear. Una vez que la especificación está completa, producir el
código es mecánico y un asistente de IA lo hace bien. Cuando la especificación está
incompleta, el asistente **elige por nosotros y no avisa que eligió**.

Por eso la actividad no empieza pidiendo código: empieza pidiendo una ficha. La ficha
completa está en [`ficha_especificacion.md`](ficha_especificacion.md).

## Procedimiento

1. Lee la ficha y escribe tú el esqueleto del código.
2. Pásale la ficha a un asistente de IA y pídele la implementación.
3. Compara los **números** con los de este cuaderno, no el estilo del código.
4. Clasifica cada diferencia: estilo / decisión no especificada / error.
5. **Anota qué le faltó a la ficha.** Ése es el hallazgo, no el código.

> **Este cuaderno es la solución de referencia.** No lo ejecutes antes de intentar la
> actividad: el ejercicio pierde sentido si se conoce la respuesta.

---

## El caso: la función de costos de Nerlove (1963)

Nerlove estimó una función de costos Cobb-Douglas para 145 empresas de generación
eléctrica en Estados Unidos en 1955, con el fin de medir si la industria presentaba
**economías de escala**. Es el ejemplo canónico de forma funcional log-log y se
reproduce en Greene (2012, cap. 10) y en Christensen y Greene (1976).

$$\ln(CT) = \beta_0 + \beta_1 \ln(Q) + \beta_2 \ln(P_L) + \beta_3 \ln(P_F) + \beta_4 \ln(P_K) + \varepsilon$$

El modelo es **lineal en los parámetros** aunque no en las variables, y por eso MCO
aplica. Esta distinción está en el capítulo 1 de las notas.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

RUTA = "../../Clase_01_RegresionLineal/nerlove63.dta"

# Valores publicados, para la verificación del final.
PUBLICADO = {"beta_output": 0.721, "n": 145}

datos = pd.read_stata(RUTA)

# La especificación es log-log.
for columna in ["totcost", "output", "plabor", "pfuel", "pkap"]:
    datos["l_" + columna] = np.log(datos[columna])

print(f"Observaciones: {len(datos)}")
datos.head()

Observaciones: 145


,totcost,output,plabor,pfuel,pkap,l_totcost,l_output,l_plabor,l_pfuel,l_pkap
0,0.082,2.0,2.09,17.900000,183.0,-2.501036,0.693147,0.737164,2.884801,5.209486
1,0.661,3.0,2.05,35.099998,174.0,-0.414001,1.098612,0.717840,3.558201,5.159055
2,0.990,4.0,2.05,35.099998,171.0,-0.010050,1.386294,0.717840,3.558201,5.141664
3,0.315,4.0,1.83,32.200001,166.0,-1.155183,1.386294,0.604316,3.471967,5.111988
4,0.197,5.0,2.12,28.600000,233.0,-1.624552,1.609438,0.751416,3.353407,5.451038


## Estimación por MCO

In [2]:
formula = "l_totcost ~ l_output + l_plabor + l_pfuel + l_pkap"
modelo = smf.ols(formula, data=datos).fit()

tabla = pd.DataFrame(
    {
        "coef": modelo.params,
        "ee": modelo.bse,
        "t": modelo.tvalues,
        "p": modelo.pvalues,
    }
)
print(f"N = {int(modelo.nobs)}   R^2 = {modelo.rsquared:.4f}\n")
tabla.round(4)

N = 145   R^2 = 0.9260



,coef,ee,t,p
Intercept,-3.5265,1.7744,-1.9875,0.0488
l_output,0.7204,0.0175,41.2445,0.0000
l_plabor,0.4363,0.2910,1.4992,0.1361
l_pfuel,0.4265,0.1004,4.2495,0.0000
l_pkap,-0.2199,0.3394,-0.6478,0.5182


## Interpretación estructural

El coeficiente del producto es la **elasticidad costo-producto**. Su recíproco mide las
economías de escala: si $1/\beta_1 > 1$, el costo crece menos que proporcionalmente
respecto del producto.

In [3]:
beta_q = modelo.params["l_output"]

print(f"Elasticidad costo-producto:     {beta_q:.4f}")
print(f"Economías de escala (1/beta_q): {1 / beta_q:.4f}")
print()
print("Al ser 1/beta_q > 1 hay economías de escala: el costo crece menos")
print("que proporcionalmente respecto del producto.")

Elasticidad costo-producto:     0.7204
Economías de escala (1/beta_q): 1.3881

Al ser 1/beta_q > 1 hay economías de escala: el costo crece menos
que proporcionalmente respecto del producto.


## Prueba de homogeneidad de grado uno en precios

Bajo la teoría, la función de costos debe ser **homogénea de grado 1 en los precios de
los insumos**: si todos los precios se duplican, el costo total se duplica. Eso implica
la restricción

$$\beta_2 + \beta_3 + \beta_4 = 1$$

que es contrastable con una prueba $F$.

In [4]:
suma = modelo.params[["l_plabor", "l_pfuel", "l_pkap"]].sum()
prueba = modelo.f_test("l_plabor + l_pfuel + l_pkap = 1")

print(f"Suma de los coeficientes de precios: {suma:.4f}")
print(f"F = {float(prueba.fvalue):.4f}   p = {float(prueba.pvalue):.4f}")

Suma de los coeficientes de precios: 0.6430
F = 0.5737   p = 0.4501


**Lo primero que hay que notar.** La suma muestral es 0.64, que está lejos de 1, y sin
embargo la prueba **no rechaza** la restricción (p = 0.45). No hay contradicción: los
errores estándar de los coeficientes de precios son grandes.

«Lejos de 1» y «estadísticamente distinto de 1» no son lo mismo, y es un error frecuente
tratarlos como si lo fueran.

## Verificación contra el resultado publicado

La regla de trabajo del curso: **un resultado que no se puede contrastar contra algo no
se reporta.** Aquí el contraste es directo, porque el artículo original está publicado.

In [5]:
diferencia = abs(beta_q - PUBLICADO["beta_output"])

print(f"beta_output estimado  = {beta_q:.4f}")
print(f"beta_output publicado = {PUBLICADO['beta_output']:.4f}   (Nerlove, 1963)")
print(f"diferencia            = {diferencia:.4f}", end="   ")
print("COINCIDE" if diferencia < 0.01 else "NO COINCIDE — revisar")

print(f"\nN estimado = {int(modelo.nobs)}, publicado = {PUBLICADO['n']}", end="   ")
print("COINCIDE" if int(modelo.nobs) == PUBLICADO["n"] else "NO COINCIDE")

beta_output estimado  = 0.7204
beta_output publicado = 0.7210   (Nerlove, 1963)
diferencia            = 0.0006   COINCIDE

N estimado = 145, publicado = 145   COINCIDE


## Lo segundo que hay que notar: un resultado inadmisible

El coeficiente del precio del capital sale **negativo**.

In [6]:
print(f"beta del precio del capital: {modelo.params['l_pkap']:.4f}")
print(f"su error estándar:           {modelo.bse['l_pkap']:.4f}")
print(f"valor p:                     {modelo.pvalues['l_pkap']:.4f}")

beta del precio del capital: -0.2199
su error estándar:           0.3394
valor p:                     0.5182


Un insumo más caro no puede **reducir** el costo total: el signo contradice la teoría de
la producción.

**No es un error de programación.** Está en el artículo original, que lo atribuye a error
de medición en el precio del capital. Christensen y Greene (1976) retoman el problema con
una forma funcional más flexible (translogarítmica).

Ésta es la moraleja de la actividad: **replicar un resultado publicado no lo vuelve
económicamente defendible**, y detectarlo es trabajo del economista, no del asistente que
escribió el código. El código es correcto; el problema está en otro lado.

## Cierre de la actividad

Vuelve a la ficha y responde:

1. ¿La especificación mencionaba qué hacer si un coeficiente sale con signo contrario al
   que predice la teoría? ¿Debería?
2. ¿Especificaba qué errores estándar usar? Si no, ¿qué eligió el asistente y lo dijo?
3. ¿Pedía la prueba de homogeneidad, o la habrías omitido?

El ejercicio de extensión de la ficha pide escribirla completa para tres casos más
—elección binaria, panel y variables instrumentales—, donde la parte difícil es siempre
la misma: **lo que la ficha no dice, alguien lo decide.**

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para el calendario de actividades.